# AnalystLab Africa – Week 3: Statistics & Probability
## Task 2: California Housing Dataset
**Author:** Kira Jotham Ishaya | **Batch B** | June–August 2025

---
**Objectives:** Descriptive statistics · Probability distributions · Hypothesis testing · Correlation vs causation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import (shapiro, normaltest, ttest_ind, f_oneway,
                          mannwhitneyu, kruskal, pearsonr, spearmanr)
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
SEED = 42; np.random.seed(SEED)
print("Libraries loaded ✓")

## 1. Load & Clean Data

In [ ]:
url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

df["total_bedrooms"].fillna(df["total_bedrooms"].median(), inplace=True)
df["rooms_per_household"]      = df["total_rooms"]    / df["households"]
df["bedrooms_per_room"]        = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"]     / df["households"]
df["log_house_value"]          = np.log1p(df["median_house_value"])
df["log_income"]               = np.log1p(df["median_income"])

print(f"Shape: {df.shape}")
display(df.head())
print("\nMissing values after imputation:")
display(df.isnull().sum()[df.isnull().sum()>0])

## 2. Descriptive Statistics

In [ ]:
numeric_cols = ["median_house_value","median_income","housing_median_age",
                "rooms_per_household","bedrooms_per_room","population_per_household"]
desc = df[numeric_cols].describe().T
desc["variance"] = df[numeric_cols].var()
desc["skewness"] = df[numeric_cols].skew()
desc["kurtosis"] = df[numeric_cols].kurtosis()
desc["cv"]       = desc["std"] / desc["mean"]
display(desc.round(3))

print("\n── Median House Value by Ocean Proximity ──")
display(df.groupby("ocean_proximity")["median_house_value"]
          .agg(["mean","median","std","count"]).round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("California Housing – Descriptive Statistics", fontsize=14, fontweight="bold")

axes[0,0].hist(df["median_house_value"], bins=40, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0,0].axvline(df["median_house_value"].mean(),   color="#DD8452", lw=2, ls="--")
axes[0,0].axvline(df["median_house_value"].median(), color="#55A868", lw=2, ls=":")
axes[0,0].set_title("Median House Value")
axes[0,0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x/1e3:.0f}k"))

axes[0,1].hist(df["log_house_value"], bins=40, color="#55A868", edgecolor="white", alpha=0.85)
axes[0,1].set_title("log(House Value+1)")

axes[0,2].hist(df["median_income"], bins=40, color="#8172B2", edgecolor="white", alpha=0.85)
axes[0,2].set_title("Median Income (×$10k)")

order = df.groupby("ocean_proximity")["median_house_value"].median().sort_values().index
vals  = [df[df["ocean_proximity"]==g]["median_house_value"].values for g in order]
axes[1,0].boxplot(vals, labels=order); axes[1,0].set_title("House Value by Ocean Proximity")
axes[1,0].tick_params(axis="x", rotation=30)
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x/1e3:.0f}k"))

axes[1,1].hist(df["housing_median_age"], bins=30, color="#DD8452", edgecolor="white", alpha=0.85)
axes[1,1].set_title("Housing Median Age")

axes[1,2].hist(df["rooms_per_household"].clip(0,20), bins=40, color="#C44E52", edgecolor="white", alpha=0.85)
axes[1,2].set_title("Rooms per Household (clipped at 20)")

plt.tight_layout(); plt.show()

## 3. Probability Distributions

In [ ]:
print("Normality Tests:")
for col in ["median_house_value","log_house_value","median_income","log_income","housing_median_age"]:
    sample = df[col].dropna().sample(min(500, len(df)), random_state=SEED)
    stat_s, p_s = shapiro(sample)
    stat_k, p_k = normaltest(df[col].dropna())
    print(f"  {col:30s}: Shapiro W={stat_s:.4f} p={p_s:.4f} | D'Agostino K²={stat_k:.4f} p={p_k:.4f}")

mu_lhv, sig_lhv = stats.norm.fit(df["log_house_value"])
print(f"\nNormal fit log(HouseValue):  μ={mu_lhv:.4f}  σ={sig_lhv:.4f}")
print("\nOcean Proximity proportions:")
display((df["ocean_proximity"].value_counts(normalize=True)*100).round(2).to_frame())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("California Housing – Probability Distributions", fontsize=14, fontweight="bold")

x_lhv = np.linspace(df["log_house_value"].min(), df["log_house_value"].max(), 300)
axes[0,0].hist(df["log_house_value"], bins=40, density=True, color="#4C72B0", alpha=0.6, label="Actual")
axes[0,0].plot(x_lhv, stats.norm.pdf(x_lhv, mu_lhv, sig_lhv), "r-", lw=2, label="Normal fit")
axes[0,0].set_title("log(House Value): Normal Fit"); axes[0,0].legend()

(osm, osr), (s, i, r) = stats.probplot(df["log_house_value"], dist="norm")
axes[0,1].scatter(osm, osr, s=4, alpha=0.4, color="#4C72B0")
axes[0,1].plot(osm, s*np.array(osm)+i, "r-", lw=2)
axes[0,1].set_title(f"Q-Q: log(House Value)  R={r:.3f}")

(osm2, osr2), (s2, i2, r2) = stats.probplot(df["median_income"], dist="norm")
axes[0,2].scatter(osm2, osr2, s=4, alpha=0.4, color="#55A868")
axes[0,2].plot(osm2, s2*np.array(osm2)+i2, "r-", lw=2)
axes[0,2].set_title(f"Q-Q: Median Income  R={r2:.3f}")

for cat in df["ocean_proximity"].value_counts().index[:4]:
    axes[1,0].hist(df[df["ocean_proximity"]==cat]["log_house_value"], bins=30, alpha=0.45, label=cat, density=True)
axes[1,0].set_title("log(HV) by Ocean Proximity"); axes[1,0].legend(fontsize=8)

mu_age, sig_age = stats.norm.fit(df["housing_median_age"])
x_age = np.linspace(df["housing_median_age"].min(), df["housing_median_age"].max(), 200)
axes[1,1].hist(df["housing_median_age"], bins=30, density=True, color="#8172B2", alpha=0.6, label="Actual")
axes[1,1].plot(x_age, stats.norm.pdf(x_age, mu_age, sig_age), "r-", lw=2, label="Normal fit")
axes[1,1].set_title(f"Housing Age (skew={df['housing_median_age'].skew():.2f})"); axes[1,1].legend()

prox_counts = df["ocean_proximity"].value_counts(normalize=True)*100
axes[1,2].bar(prox_counts.index, prox_counts.values,
              color=sns.color_palette("muted", len(prox_counts)), edgecolor="white")
axes[1,2].set_title("Ocean Proximity (%)"); axes[1,2].tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()

## 4. Hypothesis Testing

In [ ]:
alpha = 0.05

def report(name, stat, p):
    d = "✅ REJECT H₀" if p < alpha else "❌ FAIL TO REJECT H₀"
    print(f"\n{name}\n  Statistic={stat:.4f}  p={p:.6f}  → {d}")

groups = [df[df["ocean_proximity"]==g]["log_house_value"].values for g in df["ocean_proximity"].unique()]
f_stat, p_anova = f_oneway(*groups)
report("H1: log(HV) equal across ocean_proximity groups (ANOVA)", f_stat, p_anova)

inland  = df[df["ocean_proximity"]=="INLAND"]["log_house_value"]
nearOc  = df[df["ocean_proximity"]=="NEAR OCEAN"]["log_house_value"]
u1, pu1 = stats.mannwhitneyu(inland, nearOc, alternative="two-sided")
report("  Post-hoc: INLAND vs NEAR OCEAN (Mann-Whitney)", u1, pu1)

median_inc = df["median_income"].median()
high_inc = df[df["median_income"] > median_inc]["log_house_value"]
low_inc  = df[df["median_income"] <= median_inc]["log_house_value"]
t2, p2 = ttest_ind(high_inc, low_inc, equal_var=False)
report("H2: log(HV) equal for high vs low income blocks (Welch t-test)", t2, p2)

df["age_quartile"] = pd.qcut(df["housing_median_age"], 4, labels=["Q1","Q2","Q3","Q4"])
kw_groups = [df[df["age_quartile"]==q]["log_house_value"].values for q in ["Q1","Q2","Q3","Q4"]]
h_stat, p_kw = kruskal(*kw_groups)
report("H3: log(HV) equal across housing age quartiles (Kruskal-Wallis)", h_stat, p_kw)

coastal = df[df["ocean_proximity"].isin(["<1H OCEAN","NEAR OCEAN","NEAR BAY"])]["log_house_value"]
inland2 = df[df["ocean_proximity"]=="INLAND"]["log_house_value"]
t4, p4  = ttest_ind(coastal, inland2, equal_var=False)
report("H4: Coastal log(HV) = Inland log(HV)", t4, p4)

d_inc = (high_inc.mean()-low_inc.mean())/np.sqrt((high_inc.std()**2+low_inc.std()**2)/2)
d_loc = (coastal.mean()-inland2.mean())/np.sqrt((coastal.std()**2+inland2.std()**2)/2)
print(f"\nCohen's d — Income: {d_inc:.3f}  |  Location: {d_loc:.3f}  (both large)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("California Housing – Hypothesis Testing", fontsize=14, fontweight="bold")

order2 = df.groupby("ocean_proximity")["log_house_value"].median().sort_values().index.tolist()
data_box = [df[df["ocean_proximity"]==g]["log_house_value"].values for g in order2]
axes[0,0].boxplot(data_box, labels=order2)
axes[0,0].set_title(f"log(HV) by Ocean Proximity\nF={f_stat:.1f}, p<0.001")
axes[0,0].tick_params(axis="x", rotation=30)

axes[0,1].violinplot([low_inc.values, high_inc.values], positions=[0,1], showmedians=True)
axes[0,1].set_xticks([0,1]); axes[0,1].set_xticklabels(["Low Income","High Income"])
axes[0,1].set_title(f"log(HV): Income Groups  p<0.001")

df.boxplot(column="log_house_value", by="age_quartile", ax=axes[0,2])
plt.sca(axes[0,2]); plt.title(f"log(HV) by Age Quartile  KW p<0.001")

mean_hv = df.groupby("ocean_proximity")["median_house_value"].mean().sort_values()
axes[1,0].barh(mean_hv.index, mean_hv.values/1e3, color="#4C72B0", edgecolor="white")
axes[1,0].set_title("Mean House Value ($k) by Proximity"); axes[1,0].set_xlabel("$k")

axes[1,1].violinplot([inland2.values, coastal.values], positions=[0,1], showmedians=True)
axes[1,1].set_xticks([0,1]); axes[1,1].set_xticklabels(["Inland","Coastal"])
axes[1,1].set_title(f"Coastal vs Inland log(HV)  d={d_loc:.2f}")

df["age_quartile"].value_counts().sort_index().plot(kind="bar", ax=axes[1,2], color="#8172B2", edgecolor="white")
axes[1,2].set_title("Blocks per Age Quartile"); axes[1,2].tick_params(axis="x", rotation=0)

plt.tight_layout(); plt.show()

## 5. Correlation vs Causation

In [ ]:
feat_cols = ["median_house_value","median_income","housing_median_age",
             "rooms_per_household","bedrooms_per_room",
             "population_per_household","latitude","longitude"]

corr_matrix = df[feat_cols].corr(method="pearson")
print("Pearson r with median_house_value:")
display(corr_matrix["median_house_value"].sort_values(ascending=False).round(4).to_frame())

print("\nDetailed Pearson with p-values:")
for col in ["median_income","housing_median_age","rooms_per_household","latitude","longitude"]:
    r, p = pearsonr(df[col], df["median_house_value"])
    print(f"  {col:28s}: r={r:+.4f}  p={p:.4e}")

r_lat_inc, _ = pearsonr(df["latitude"], df["median_income"])
print(f"\nSpurious example: latitude vs income r={r_lat_inc:.4f}")
print("  → Both correlate with house value, but latitude doesn't cause income.")

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.suptitle("California Housing – Correlation Analysis", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
ax1=fig.add_subplot(gs[0,:2]); ax2=fig.add_subplot(gs[0,2])
ax3=fig.add_subplot(gs[1,0]);  ax4=fig.add_subplot(gs[1,1]); ax5=fig.add_subplot(gs[1,2])

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            ax=ax1, linewidths=0.5, annot_kws={"size":8})
ax1.set_title("Pearson Correlation Matrix")

sample = df.sample(3000, random_state=SEED)
ax2.scatter(sample["median_income"], sample["median_house_value"]/1e3, alpha=0.3, s=8)
m, b = np.polyfit(df["median_income"], df["median_house_value"]/1e3, 1)
x_l = np.linspace(df["median_income"].min(), df["median_income"].max(), 100)
ax2.plot(x_l, m*x_l+b, "r-", lw=2)
r_im, _ = pearsonr(df["median_income"], df["median_house_value"])
ax2.set_title(f"Income vs House Value  r={r_im:.3f}"); ax2.set_xlabel("Income"); ax2.set_ylabel("$k")

feat_corr = corr_matrix["median_house_value"].drop("median_house_value").sort_values()
bar_colors = ["#C44E52" if v<0 else "#55A868" for v in feat_corr.values]
ax3.barh(feat_corr.index, feat_corr.values, color=bar_colors, edgecolor="white")
ax3.axvline(0, color="black", lw=0.8); ax3.set_title("Pearson r with House Value")

sc = ax4.scatter(sample["longitude"], sample["latitude"],
                 c=sample["median_house_value"], cmap="viridis", alpha=0.5, s=6)
plt.colorbar(sc, ax=ax4); ax4.set_title("Geographic House Value Distribution")

bpr = df["bedrooms_per_room"].clip(0,1)
ax5.scatter(bpr.sample(3000,random_state=SEED), df["median_house_value"].sample(3000,random_state=SEED)/1e3,
            alpha=0.3, s=8, color="#8172B2")
r_bpr, _ = pearsonr(bpr.dropna(), df.loc[bpr.dropna().index,"median_house_value"])
ax5.set_title(f"Bedrooms/Room vs HV  r={r_bpr:.3f}")

plt.tight_layout(); plt.show()

## 6. Key Insights Summary

| # | Finding | Evidence |
|---|---------|----------|
| 1 | **Median income is the strongest predictor** (r=0.688) | Pearson, p<0.001 |
| 2 | **Coastal proximity adds ~$115k vs Inland** | Cohen's d=1.47, p<0.001 |
| 3 | **House value is right-skewed; log-transform is necessary** | Shapiro p<0.001; skew reduces 0.98→~0 |
| 4 | **Latitude correlates with value** but only as a geographic proxy | Spurious via coastal-city clustering |
| 5 | **Bedrooms/room (r=−0.26)**: denser occupancy → lower value, but this reflects poverty, not geometry | Confounding variable |

> **Key Caution:** `total_rooms`, `households`, and `population` are highly multicollinear. Using them together in regression inflates apparent correlations — engineered ratios (rooms_per_household) are better features.
